# 接触如何破坏一阶梯度：无人机(光滑) vs 四足最小代理(SLIP/Hopper)

> 配套研究笔记：[`research_note.md`](research_note.md)　|　全部实验脚本：[`run_experiments.py`](run_experiments.py)　|　动力学模块：[`slip_dynamics.py`](slip_dynamics.py)

**目标**：用 1 自由度**垂直 Hopper**（SLIP 退化形式）作为四足"足端接触切换"的最小不可约模型，
干净隔离"接触如何破坏可微训练所需的一阶梯度(FoG)"，并与无人机式**光滑、可微平坦**动力学对照。

**核心结论预告**：接触让 BPTT 梯度幅值爆炸 ~4000×，并让约一半的梯度**方向错误**；
而梯度衰减/裁剪（无人机 GCGL 的手段）只能**压幅值、不能纠方向**。

> 运行环境：conda `pytorch`，PyTorch 2.4.1，float64。整本约 1 分钟。

In [ ]:
%matplotlib inline
import torch, numpy as np, matplotlib.pyplot as plt
import slip_dynamics as sd

torch.set_default_dtype(torch.float64)          # 梯度精度微实验 -> float64
dev = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('device =', dev, '| reuse drone g_decay =', sd.HAVE_DRONE_GDECAY)

C_BASE, C_CONTACT, C_SMOOTH, C_HARD, C_STOCH = '#4C72B0','#C44E52','#55A868','#8C8C8C','#64B5CD'
plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'grid.alpha':0.3})

## E1 接触力定律及其解析梯度（复现 DiffSim2Real 2024, Fig.2）

四种法向力模型作为穿透深度 $d$ 的函数（$d>0$=接触中）。看**右图的梯度**：
硬接触/随机平滑的一阶梯度**几乎处处为 0**（无信息），只有**解析平滑**给出有界、有信息的钟形梯度。

In [ ]:
d = torch.linspace(-0.04, 0.04, 1601, device=dev)
specs = [('hard',C_HARD,'Hard  f=F0·H(d)'),('soft',C_BASE,'Soft  f=relu(k·d)'),
         ('stoch',C_STOCH,'Stochastic-smoothed (FoG≡0)'),('smooth',C_SMOOTH,'Analytic-smooth f=F0·σ(d/ε)')]
fig, ax = plt.subplots(1,2,figsize=(11,3.6))
for mdl,c,lab in specs:
    f,gr = sd.contact_force_law(d, mdl, F0=50., k=4000., eps=0.006, sigma=0.012)
    ax[0].plot(d.cpu()*1e3, f.cpu(), color=c, lw=2, label=lab)
    ax[1].plot(d.cpu()*1e3, gr.cpu(), color=c, lw=2, label=lab)
for a in ax: a.axvline(0,color='k',lw=.8,ls=':'); a.set_xlabel('penetration d [mm]')
ax[0].set_ylim(-2,80); ax[0].set_title('(a) contact force $f_n$'); ax[0].set_ylabel('force [N]')
ax[1].set_ylim(-100,2200); ax[1].set_title(r'(b) first-order gradient $\partial f_n/\partial d$'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
print('hard |grad|max =', float(sd.contact_force_law(d,'hard')[1].abs().max()),
      '| smooth |grad|max =', round(float(sd.contact_force_law(d,'smooth',F0=50.,eps=0.006)[1].abs().max()),1))

## E2 可微 Hopper 弹跳轨迹（物理自检）

软/刚/解析平滑三种接触都给出正确的"自由落体→触地→回弹→衰减"，确认仿真可信。

In [ ]:
y0 = torch.tensor(1.2, device=dev); v0 = torch.tensor(0.0, device=dev)
n, dt = 4000, 1e-3
fig, ax = plt.subplots(1,2,figsize=(11,3.4))
for mdl,c,kn in [('soft',C_BASE,4000.),('stiff',C_CONTACT,30000.),('smooth',C_SMOOTH,4000.)]:
    Y,V = sd.rollout_hopper(y0,v0,n,dt=dt,model=mdl,k_n=kn,k_d=12.,eps=0.008)
    t = np.arange(n+1)*dt
    ax[0].plot(t, Y.cpu(), color=c, lw=1.5, label=f'{mdl} (k_n={kn:.0f})')
    ax[1].plot(Y.cpu(), V.cpu(), color=c, lw=.9, alpha=.8, label=mdl)
ax[0].axhline(0,color='k',lw=.8,ls=':'); ax[0].set_xlabel('t [s]'); ax[0].set_ylabel('height y [m]'); ax[0].legend(fontsize=8)
ax[1].axvline(0,color='k',lw=.8,ls=':'); ax[1].set_xlabel('y [m]'); ax[1].set_ylabel('v [m/s]'); ax[1].set_title('phase portrait')
plt.tight_layout(); plt.show()

## E3 + E4 头条：接触粗糙化景观 + 注入**方向性**梯度偏差

扫描初始参数（无人机基线=净加速度指令；Hopper=初速度 $v_0$），一次 batched rollout 同时得到：
- **景观**（终端成本曲线）→ 三阶差分量化非光滑度；
- **解析 FoG vs 有限差分"真梯度"** → 平均偏差 + **符号不一致率**（方向错误率，尺度无关）。

In [ ]:
y_target = 0.6
v0_grid = torch.linspace(-1.0, 1.0, 801, device=dev)

def eval_config(mdl, kn, eps, grad_decay=1.0):
    if mdl == 'baseline':
        u = (v0_grid*8.0).clone().requires_grad_(True)
        Y = sd.rollout_smooth_pointmass(torch.full_like(v0_grid,1.2), torch.zeros_like(v0_grid), u, 150, dt=0.02)
        cost = (Y[-1]-y_target)**2; cost.sum().backward()
        ana, x = u.grad.detach(), u.detach()
    else:
        v = v0_grid.clone().requires_grad_(True)
        Y,_ = sd.rollout_hopper(torch.full_like(v0_grid,1.2), v, 3000, dt=1e-3, model=mdl,
                                k_n=kn, k_d=12., eps=eps, grad_decay=grad_decay)
        cost = (Y[-1]-y_target)**2; cost.sum().backward()
        ana, x = v.grad.detach(), v0_grid
    cost_d = cost.detach()
    fd = (cost_d[2:]-cost_d[:-2])/(x[2:]-x[:-2])
    ana_in = ana[1:-1]
    mask = fd.abs() > 1.0
    sign_dis = float(((torch.sign(ana_in)!=torch.sign(fd)) & mask).sum())/float(mask.sum()+1e-9)
    return cost_d, ana_in, fd, x, sd.landscape_roughness(cost_d), float((ana_in-fd).abs().mean()), sign_dis

configs = [('smooth-baseline(drone)','baseline',None,None,1.0,C_BASE),
           ('hopper-smooth','smooth',4000.,0.008,1.0,C_SMOOTH),
           ('hopper-soft','soft',4000.,0.008,1.0,'#DD8452'),
           ('hopper-stiff','stiff',30000.,0.008,1.0,C_CONTACT),
           ('hopper-stiff+gdecay0.6','stiff',30000.,0.008,0.6,'#8172B3')]

print(f"{'config':28s}{'roughness':>12s}{'|ana-fd|':>12s}{'dir-error':>11s}")
fig, ax = plt.subplots(1,2,figsize=(11,3.8))
for name,mdl,kn,eps,gd,c in configs:
    cost_d, ana_in, fd, x, rough, bias, sdis = eval_config(mdl,kn,eps,gd)
    print(f"{name:28s}{rough:12.4g}{bias:12.4g}{sdis:11.3f}")
    if gd==1.0:
        ax[0].plot(x.cpu(), cost_d.cpu(), color=c, lw=1.5, label=name)
    if name in ('smooth-baseline(drone)','hopper-stiff'):
        ax[1].plot(x.cpu()[1:-1], ana_in.cpu(), color=c, lw=1.3, label=f'{name} FoG')
        ax[1].plot(x.cpu()[1:-1], fd.cpu(), color=c, lw=1.0, ls='--', alpha=.7, label=f'{name} finite-diff')
ax[0].set_title('(a) loss landscape: smooth vs contact'); ax[0].set_xlabel('swept param'); ax[0].set_ylabel('terminal cost'); ax[0].legend(fontsize=7); ax[0].set_ylim(0,0.6)
ax[1].set_title('(b) FoG vs finite-diff'); ax[1].set_xlabel('swept param'); ax[1].set_ylabel('dCost/dparam'); ax[1].set_ylim(-60,60); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

**读表**：光滑基线与 hopper-smooth 的方向错误率 = **0%**；hopper-stiff ≈ **52%**（近乎抛硬币）。
关键看最后两行：`hopper-stiff` 与 `hopper-stiff+gdecay0.6` 的**方向错误率完全相同**——
因为梯度衰减是正标量缩放，$\operatorname{sign}(c\cdot g)=\operatorname{sign}(g)$，**压幅不压偏**。

## E5 BPTT 梯度爆炸 vs 视野长度（grad_decay 压幅）

$|\partial y_T/\partial v_0|$ 随 rollout 步数变化：光滑基线**有界**，接触**指数爆炸**，`grad_decay` 把幅值压下去
（这正是无人机 GDecay / Song 2024 状态对齐 α 的作用）。

In [ ]:
horizons = [int(h) for h in np.linspace(200, 6000, 25)]
def gnorm(n, mdl, kn, eps, gd=1.0, baseline=False):
    v = torch.tensor(0.3, device=dev, requires_grad=True)
    if baseline:
        Y = sd.rollout_smooth_pointmass(torch.tensor(1.2,device=dev), v, torch.tensor(0.,device=dev), n, dt=1e-3)
    else:
        Y,_ = sd.rollout_hopper(torch.tensor(1.2,device=dev), v, n, dt=1e-3, model=mdl, k_n=kn, k_d=12., eps=eps, grad_decay=gd)
    return abs(float(torch.autograd.grad(Y[-1], v)[0]))

series = {'smooth point-mass (no contact)':(C_BASE,dict(mdl='',kn=0,eps=0,baseline=True)),
          'hopper-stiff (k_n=3e4)':(C_CONTACT,dict(mdl='stiff',kn=30000.,eps=.008)),
          'hopper-soft (k_n=4e3)':('#DD8452',dict(mdl='soft',kn=4000.,eps=.008)),
          'hopper-smooth':(C_SMOOTH,dict(mdl='smooth',kn=4000.,eps=.008)),
          'hopper-stiff + grad_decay=0.6':('#8172B3',dict(mdl='stiff',kn=30000.,eps=.008,gd=0.6))}
plt.figure(figsize=(7.5,4.2))
for name,(c,kw) in series.items():
    ys=[gnorm(n,**kw) for n in horizons]
    plt.plot(horizons, ys, color=c, lw=2, marker='o' if 'no contact' in name else 's', ms=3, label=name)
plt.yscale('log'); plt.xlabel('rollout horizon (#steps, dt=1e-3)')
plt.ylabel(r'$|\partial y_T/\partial v_0|$ (log)'); plt.legend(fontsize=8)
plt.title('BPTT gradient: bounded for smooth, blows up through contact'); plt.tight_layout(); plt.show()

## 结论（详见 [`research_note.md`](research_note.md) §7 验收问答）

1. 无人机可微框架靠**光滑 + 可微平坦** ⇒ 一阶梯度**有界且方向正确**（基线方向错误率 0%）。
2. 四足的**接触**同时破坏这两点 ⇒ FoG **爆炸（×4000）且方向错误（~52%）**。
3. 无人机 GCGL 的**目标损失可直接迁移**；**梯度裁剪/衰减只能压幅、不能纠偏**（符号不一致率不变）。
4. 迁移主线：**SRBD + 解析平滑接触 + 预定步态 + 短视野/状态对齐(α=GDecay)**（Song 2024 已真机验证）。